# 10 — Load and visualize model results

This notebook reads only the result files written by notebook 09. It does not load raw data or refit a model. Figures use grayscale-safe APA 7 styling and contain no in-figure title. Add the figure number, italicized title, and note in the thesis document.


## 1. Paths


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

PROJECT_DIR=Path.cwd().resolve()
if PROJECT_DIR.name=="ANAL": PROJECT_DIR=PROJECT_DIR.parent
RESULT_DIR=PROJECT_DIR/"ANAL"/"data"/"models"
FIGURE_DIR=PROJECT_DIR/"FIG"/"model_results"
FIGURE_DIR.mkdir(parents=True,exist_ok=True)
print(f"Reading results from: {RESULT_DIR}")
print(f"Writing figures to: {FIGURE_DIR}")


## 2. Check and load result files


In [ ]:
required=["model_run_manifest.json","founding_nb2_results.csv","founding_nb2_all_terms.csv",
          "founding_nb2_count_fit.csv","survival_cox_results.csv","survival_schoenfeld_screen.csv",
          "survival_km_by_sector.csv","survival_annual_exit_rates.csv"]
missing=[name for name in required if not (RESULT_DIR/name).exists()]
if missing: raise FileNotFoundError(f"Missing result files: {missing}")
manifest=json.loads((RESULT_DIR/"model_run_manifest.json").read_text(encoding="utf-8"))
print(f"Model run: {manifest['created_at_utc']}")
print(f"Study period: {manifest['study_period']['start_year']}–{manifest['study_period']['end_year']}")


## 3. APA 7 figure style


In [ ]:
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white","savefig.facecolor":"white",
 "font.family":"sans-serif","font.sans-serif":["Arial","DejaVu Sans"],"font.size":10,
 "axes.labelsize":10,"axes.linewidth":.8,"axes.spines.top":False,"axes.spines.right":False,
 "legend.frameon":False,"legend.fontsize":9,"xtick.labelsize":9,"ytick.labelsize":9})
LINE_STYLES=["-","--","-.",":",(0,(5,1)),(0,(3,1,1,1)),(0,(1,1))]
MARKERS=["o","s","^","D","v","P","X"]
print("APA figure style loaded.")


## 4. Founding-model coefficient figure


In [ ]:
founding=pd.read_csv(RESULT_DIR/"founding_nb2_results.csv",index_col="term")
labels={"log_own_firms":"Lagged firms in cell","log_own_pop":"Population in cell, previous year",
"log_pop_access_ring_0_15":"Reachable mass, car 0–15 min","log_firms_relative_car_ring_0_15":"Relative firm density, car 0–15 min",
"log_walk_pop_ring_0_10":"Reachable mass, walk 0–10 min","log_firms_relative_walk_ring_0_10":"Relative firm density, walk 0–10 min",
"population_growth_yoy":"Year-on-year population growth","log_tt_motorway_exit":"Travel time to motorway exit",
"walk_pt_routes_10min":"PT routes within 10 min walk","pt_ohne_haltestelle":"No PT stop reachable"}
plot=founding.loc[[term for term in labels if term in founding.index]].iloc[::-1]
print(f"Input table: {plot.shape[0]} terms × {plot.shape[1]} columns")
fig,ax=plt.subplots(figsize=(8.5,.48*len(plot)+1.6)); y=np.arange(len(plot))
ax.errorbar(plot.irr,y,xerr=[plot.irr-plot.irr_ci_lower,plot.irr_ci_upper-plot.irr],fmt="o",capsize=3,
            color="black",markerfacecolor="white",linewidth=1)
ax.axvline(1,color=".45",linestyle="--",linewidth=.8); ax.set_xscale("log")
ax.set_yticks(y,[labels[t] for t in plot.index]); ax.set_xlabel("Incidence-rate ratio (95% cell-clustered CI; log scale)")
fig.tight_layout(); output=FIGURE_DIR/"founding_coefficients.png"; fig.savefig(output,dpi=300,bbox_inches="tight"); plt.show()
print(f"Saved: {output.name}")


## 5. Founding period effects


In [ ]:
periods=pd.read_csv(RESULT_DIR/"founding_nb2_all_terms.csv",index_col="term")
periods=periods.loc[periods.index.str.startswith("period_")].copy(); periods["period"]=periods.index.str.removeprefix("period_"); periods=periods.sort_values("period")
print(f"Input table: {len(periods)} quarters")
fig,ax=plt.subplots(figsize=(11,4)); ax.plot(periods.period,periods.irr,"o-",color="black",markerfacecolor="white",markersize=3,linewidth=1)
ax.fill_between(periods.period,periods.irr_ci_lower,periods.irr_ci_upper,color=".85",linewidth=0); ax.axhline(1,color=".45",linestyle="--",linewidth=.8)
ax.tick_params(axis="x",rotation=90,labelsize=7); ax.set_ylabel("IRR relative to the first analysis quarter")
fig.tight_layout(); output=FIGURE_DIR/"founding_period_effects.png"; fig.savefig(output,dpi=300,bbox_inches="tight"); plt.show(); print(f"Saved: {output.name}")


## 6. Observed and modelled founding counts


In [ ]:
count_fit=pd.read_csv(RESULT_DIR/"founding_nb2_count_fit.csv"); print(f"Input table: {count_fit.shape[0]} count groups × {count_fit.shape[1]} columns")
fig,ax=plt.subplots(figsize=(7.5,4.2)); x=np.arange(len(count_fit)); width=.38
ax.bar(x-width/2,count_fit.observed_share,width,label="Observed",color="white",edgecolor="black",linewidth=.8,hatch="///")
ax.bar(x+width/2,count_fit.modelled_share,width,label="NB2",color=".60",edgecolor="black",linewidth=.8)
ax.set_yscale("log"); ax.set_xticks(x,count_fit.births); ax.set_xlabel("Births per cell-quarter"); ax.set_ylabel("Share (log scale)"); ax.legend()
fig.tight_layout(); output=FIGURE_DIR/"founding_count_fit.png"; fig.savefig(output,dpi=300,bbox_inches="tight"); plt.show(); print(f"Saved: {output.name}")


## 7. Survival-model coefficient figure


In [ ]:
survival=pd.read_csv(RESULT_DIR/"survival_cox_results.csv",index_col="term")
labels={"log_own_pop":"Population in cell","log_own_same":"Same-Fachgruppe firms in cell","log_own_other":"Other firms in cell",
"log_pop_ring_0_15":"Reachable population mass, 0–15 min","log_same_relative_ring_0_15":"Relative same-group density, 0–15 min",
"log_other_relative_ring_0_15":"Relative other-group density, 0–15 min","log_tt_motorway_exit":"Travel time to motorway exit",
"walk_pt_routes_10min":"PT routes within 10 min walk","pt_ohne_haltestelle":"No PT stop reachable","calendar_year":"Calendar-year trend"}
plot=survival.loc[[term for term in labels if term in survival.index]].iloc[::-1]; print(f"Input table: {plot.shape[0]} terms × {plot.shape[1]} columns")
fig,ax=plt.subplots(figsize=(8.5,.48*len(plot)+1.6)); y=np.arange(len(plot))
ax.errorbar(plot.hazard_ratio,y,xerr=[plot.hazard_ratio-plot.hr_ci_lower,plot.hr_ci_upper-plot.hazard_ratio],fmt="o",capsize=3,color="black",markerfacecolor="white",linewidth=1)
ax.axvline(1,color=".45",linestyle="--",linewidth=.8); ax.set_xscale("log"); ax.set_yticks(y,[labels[t] for t in plot.index]); ax.set_xlabel("Exit hazard ratio (95% cell-clustered CI; log scale)")
fig.tight_layout(); output=FIGURE_DIR/"survival_coefficients.png"; fig.savefig(output,dpi=300,bbox_inches="tight"); plt.show(); print(f"Saved: {output.name}")


## 8. Proportional-hazards screen


In [ ]:
screen=pd.read_csv(RESULT_DIR/"survival_schoenfeld_screen.csv").sort_values("spearman_rho"); print(f"Input table: {len(screen)} covariates")
colors=np.where(screen.screen_p_value<.05,".35",".80"); fig,ax=plt.subplots(figsize=(8,.45*len(screen)+1.5))
ax.barh(np.arange(len(screen)),screen.spearman_rho,color=colors,edgecolor="black",linewidth=.6); ax.axvline(0,color="black",linewidth=.8)
ax.set_yticks(np.arange(len(screen)),[labels.get(t,t) for t in screen.term]); ax.set_xlabel("Spearman correlation: Schoenfeld residual vs event age")
ax.legend(handles=[Patch(facecolor=".35",edgecolor="black",label="Unadjusted p < .05"),Patch(facecolor=".80",edgecolor="black",label="Unadjusted p ≥ .05")])
fig.tight_layout(); output=FIGURE_DIR/"survival_ph_screen.png"; fig.savefig(output,dpi=300,bbox_inches="tight"); plt.show(); print(f"Saved: {output.name}")


## 9. Kaplan–Meier curves by sector


In [ ]:
km=pd.read_csv(RESULT_DIR/"survival_km_by_sector.csv"); print(f"Input table: {len(km):,} curve points for {km.sparte.nunique()} sectors")
fig,ax=plt.subplots(figsize=(9,5.5))
for i,((_,name),group) in enumerate(km.groupby(["sparte","sparte_name"])):
    group=group.sort_values("time_years"); ax.step(group.time_years,group.survival_probability,where="post",label=name,color="black",linestyle=LINE_STYLES[i%len(LINE_STYLES)],linewidth=1.2)
ax.set_xlim(left=0); ax.set_ylim(0,1); ax.set_xlabel("Location age (years)"); ax.set_ylabel("Survival probability"); ax.legend(fontsize=8)
fig.tight_layout(); output=FIGURE_DIR/"survival_km_by_sector.png"; fig.savefig(output,dpi=300,bbox_inches="tight"); plt.show(); print(f"Saved: {output.name}")


## 10. Annual exit rates by sector


In [ ]:
annual=pd.read_csv(RESULT_DIR/"survival_annual_exit_rates.csv"); print(f"Input table: {len(annual)} sector-year rows")
fig,ax=plt.subplots(figsize=(10,5))
for i,((_,name),group) in enumerate(annual.groupby(["sparte","sparte_name"])):
    ax.plot(group.year,group.exit_rate,marker=MARKERS[i%len(MARKERS)],linestyle=LINE_STYLES[i%len(LINE_STYLES)],color="black",markerfacecolor="white",markersize=4,linewidth=1,label=name)
ax.axvspan(2020,2021,color=".90",zorder=0); ax.set_xlabel("Calendar year"); ax.set_ylabel("Exits / locations at risk"); ax.legend(fontsize=8)
fig.tight_layout(); output=FIGURE_DIR/"survival_annual_exit_rates.png"; fig.savefig(output,dpi=300,bbox_inches="tight"); plt.show(); print(f"Saved: {output.name}")


## 11. Interpretation reminders

- The founding mass/density coefficients are reparameterized; use `founding_nb2_decomposition.csv` for separate effects.
- A hazard ratio below one means a lower exit hazard, not a percentage increase in survival time.
- The Schoenfeld plot is a diagnostic screen, not an automatic accept/reject rule.
